In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.dates as mdates

# 1. Configuration & Directories
save_dir = r"C:\Users\Ranuga\Data Science Project\4. Data Visualization\USD_Diesel_Correlation\Charts"
os.makedirs(save_dir, exist_ok=True)

# 2. Load Data
file_path = r"C:\Users\Ranuga\Data Science Project\3. Data Preprocessing\3.7 - Combining Datasets\Outputs\Final_Combined_data.csv"
df = pd.read_csv(file_path)

# Convert week strings to numbers and create a continuous Datetime column
df['week_num'] = df['week'].str.replace('w', '', regex=False).astype(int)
df['Date'] = pd.to_datetime(df['year'].astype(str) + '-' + df['week_num'].astype(str) + '-1', format='%Y-%W-%w')

# Get unique vegetable types
vegetables = df['vegetable_type'].dropna().unique()
print(f"Found {len(vegetables)} vegetable types. Generating charts...")

# 3. Generate and Save Charts per Vegetable
for veg in vegetables:
    # Filter for the specific vegetable
    df_veg = df[df['vegetable_type'] == veg]
    
    # Group by Date to get the weekly average across different markets
    weekly_data = df_veg.groupby('Date').agg({
        'usd_exchange_rate': 'mean',
        'lanka_auto_diesel_price': 'mean',
        'retail_price': 'mean',
        'mean_farmer_price': 'mean'
    }).reset_index().sort_values('Date')
    
    # Drop rows where all values are NaN just in case
    weekly_data = weekly_data.dropna(subset=['usd_exchange_rate', 'lanka_auto_diesel_price', 'retail_price', 'mean_farmer_price'], how='all')
    
    fig, ax1 = plt.subplots(figsize=(16, 8))
    
    # --- Axis 1: Vegetable Prices (Left) ---
    color_retail = 'tab:green'
    color_farmer = 'tab:orange'
    
    ax1.set_xlabel('Time (Year)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Vegetable Price (LKR / kg)', fontsize=12, fontweight='bold')
    
    line1 = ax1.plot(weekly_data['Date'], weekly_data['retail_price'], color=color_retail, label='Retail Price', linewidth=2, alpha=0.9)
    line2 = ax1.plot(weekly_data['Date'], weekly_data['mean_farmer_price'], color=color_farmer, label='Farmer Price', linewidth=2, alpha=0.9)
    ax1.tick_params(axis='y')

    # --- Axis 2: Macroeconomic Indicators (Right) ---
    ax2 = ax1.twinx()  
    color_usd = 'tab:blue'
    color_diesel = 'tab:red'
    
    ax2.set_ylabel('Macro Indicators (LKR)', fontsize=12, fontweight='bold')  
    line3 = ax2.plot(weekly_data['Date'], weekly_data['usd_exchange_rate'], color=color_usd, label='USD Exchange Rate', linewidth=2, linestyle='--')
    line4 = ax2.plot(weekly_data['Date'], weekly_data['lanka_auto_diesel_price'], color=color_diesel, label='Diesel Price', linewidth=2, linestyle=':')
    ax2.tick_params(axis='y')
    
    # Combine legends into one box
    lines = line1 + line2 + line3 + line4
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left', framealpha=0.9, fontsize=11)
    
    # Format x-axis continuously
    ax1.xaxis.set_major_locator(mdates.YearLocator())
    ax1.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=(4, 7, 10)))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.xticks(rotation=45)
    
    # Set dynamic title
    plt.title(f'Macroeconomic Indicators vs Retail & Farmer Prices: {str(veg).title()}', fontsize=16, fontweight='bold')
    fig.tight_layout()  
    plt.grid(alpha=0.3)
    
    # Save figure to the Charts folder
    safe_veg_name = str(veg).replace('/', '_').replace('\\', '_')
    save_path = os.path.join(save_dir, f"{safe_veg_name}_trends.png")
    plt.savefig(save_path, dpi=300)
    
    # Display the plot in the Notebook
    plt.show()

plt.show()
print(f"All charts successfully saved to:\n{save_dir}")

ModuleNotFoundError: No module named 'pandas'